In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry

from emu_renewal.inputs import DATA_PATH
from emu_renewal.outputs import get_output_dir, load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
country = "France"
analysis_times = {
    "no_mob": "20250121_2159",
    "google_nonresi_linear": "20250121_2116",
    "google_nonresi_square": "20250121_2120",
    "fb_linear": "20250121_2133",
    "fb_square": "20250121_2146",
}

In [ ]:
spaghs = {}
targets = {}
idatas = {}
mobs = {}
for analysis, time in analysis_times.items():
    outdir = get_output_dir(country, analysis, time)
    spaghs[analysis] = pd.read_hdf(outdir / "spaghetti.h5")
    targets[analysis] = load_targets(country, analysis, time)
    idatas[analysis] = az.from_netcdf(outdir / "idata.nc")
    mobs[analysis] = pd.read_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_2}_mob_data.csv", index_col=0)

In [ ]:
from plotly.subplots import make_subplots
from matplotlib import pyplot as plt

In [ ]:
targ_names = targets["no_mob"].keys()
fig, axes = plt.subplots(len(targ_names), len(analysis_times), figsize=(10, 8), sharex=True)
fig.tight_layout()
out_style = {"color": "black", "width": 0.5}
targ_style = {"color": "red"}
for a, analysis in enumerate(analysis_times):
    for o, out in enumerate(targ_names):
        ax = axes[a, o]
        for col in spaghs[analysis][out].columns:
            ax.plot(spaghs[analysis].index, spaghs[analysis][out][col])
        target = targets[analysis][out]
        target_scatter = ax.plot(target.index, target)

In [ ]:
priors = pickle.load(open(outdir / "priors.pkl", "rb"))
epi_params = [p for p in priors if p != "shared_dispersion"]
plot_post_prior_comparison(idata, epi_params, priors, req_grid=[8, 2], req_size=[10, 40]);